In [4]:
import pandas as pd
!pip install us
import us

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 360.5/360.5 kB 8.2 MB/s eta 0:00:00


In [5]:
#read the data

raw = pd.read_csv(
    "/content/Average_retail_price_of_electricity (1).csv",
    skiprows=4
)

raw.head()

FileNotFoundError: [Errno 2] No such file or directory: 'data/electricity/Average_retail_price_of_electricity.csv'

In [ ]:
#cleaning unnecessary top rows
electricity = raw[
    raw["description"]
    .str.startswith("Residential :", na=False)
].copy()

electricity.head()

,description,units,source key,Jan 2001,Feb 2001,Mar 2001,Apr 2001,May 2001,Jun 2001,Jul 2001,...,Aug 2025,Sep 2025,Oct 2025,Nov 2025,Dec 2025,Jan 2026,Feb 2026,Mar 2026,Apr 2026,May 2026
3,Residential : United States,cents per kilowatthour,ELEC.PRICE.US-RES.M,7.73,8.04,8.32,8.46,8.83,9.07,9.03,...,17.61,18.08,17.97,17.78,17.24,17.45,17.65,18.56,18.83,18.44
4,Residential : New England,cents per kilowatthour,ELEC.PRICE.NEW-RES.M,11.80,11.72,11.82,12.17,12.20,12.19,12.50,...,29.07,29.54,29.22,28.86,28.37,29.36,29.91,29.42,29.49,28.14
5,Residential : Connecticut,cents per kilowatthour,ELEC.PRICE.CT-RES.M,10.70,10.23,10.60,10.87,11.20,11.06,11.00,...,30.24,30.48,27.72,27.02,25.30,28.30,30.77,30.47,32.24,27.37
6,Residential : Maine,cents per kilowatthour,ELEC.PRICE.ME-RES.M,12.96,12.84,12.64,12.32,14.24,13.31,13.01,...,28.25,27.98,26.63,27.85,30.39,30.73,32.17,28.32,28.42,28.63
7,Residential : Massachusetts,cents per kilowatthour,ELEC.PRICE.MA-RES.M,11.86,12.03,12.08,12.49,12.47,12.57,13.24,...,30.61,30.41,31.37,31.22,30.88,31.16,30.46,30.21,29.45,28.82


In [ ]:
#extracting US states


valid_states = [s.name for s in us.states.STATES]
valid_states.append("United States")


electricity["State"] = (
    electricity["description"]
    .str.replace("Residential :", "", regex=False)
    .str.strip()
)

electricity = electricity[
    electricity["State"].isin(valid_states)
].copy()

electricity.head()

,description,units,source key,Jan 2001,Feb 2001,Mar 2001,Apr 2001,May 2001,Jun 2001,Jul 2001,...,Sep 2025,Oct 2025,Nov 2025,Dec 2025,Jan 2026,Feb 2026,Mar 2026,Apr 2026,May 2026,State
3,Residential : United States,cents per kilowatthour,ELEC.PRICE.US-RES.M,7.73,8.04,8.32,8.46,8.83,9.07,9.03,...,18.08,17.97,17.78,17.24,17.45,17.65,18.56,18.83,18.44,United States
5,Residential : Connecticut,cents per kilowatthour,ELEC.PRICE.CT-RES.M,10.70,10.23,10.60,10.87,11.20,11.06,11.00,...,30.48,27.72,27.02,25.30,28.30,30.77,30.47,32.24,27.37,Connecticut
6,Residential : Maine,cents per kilowatthour,ELEC.PRICE.ME-RES.M,12.96,12.84,12.64,12.32,14.24,13.31,13.01,...,27.98,26.63,27.85,30.39,30.73,32.17,28.32,28.42,28.63,Maine
7,Residential : Massachusetts,cents per kilowatthour,ELEC.PRICE.MA-RES.M,11.86,12.03,12.08,12.49,12.47,12.57,13.24,...,30.41,31.37,31.22,30.88,31.16,30.46,30.21,29.45,28.82,Massachusetts
8,Residential : New Hampshire,cents per kilowatthour,ELEC.PRICE.NH-RES.M,13.52,13.02,13.08,13.93,11.58,12.08,12.66,...,27.82,27.27,27.37,26.28,26.32,26.52,26.92,27.24,27.33,New Hampshire


In [ ]:

# remove DC
electricity = electricity[
    electricity["State"] != "District Of Columbia"
].copy()
"District Of Columbia" in electricity["State"].values

False

In [ ]:
# all date columns
date_cols = electricity.columns[3:-1]   # exclude State

map_rows = []

for _, row in electricity.iterrows():

    state = row["State"]

    # latest non-null value
    values = row[date_cols]

    latest_col = values.last_valid_index()
    latest_value = values[latest_col]

    remarks = ""

    # only write a remark if latest month isn't the dataset's last month
    if latest_col != date_cols[-1]:
        remarks = f"{latest_col} used"

    map_rows.append([
        state,
        latest_value,
        remarks
    ])

map_table = pd.DataFrame(
    map_rows,
    columns=["Name", "Values", "Remarks"]
)

In [ ]:
map_table['Name'].count()

np.int64(51)

In [ ]:
#sort by electricity price
map_table = (
    map_table
    .sort_values("Values", ascending=False)
    .reset_index(drop=True)
)

#display(map_table.style.hide(axis="index"))
map_table

,Name,Values,Remarks
0,Hawaii,52.00,
1,California,33.25,
2,New York,29.93,
3,Rhode Island,29.46,
4,Massachusetts,28.82,
5,Maine,28.63,
6,Alaska,28.23,
7,Connecticut,27.37,
8,New Hampshire,27.33,
9,Vermont,24.89,


In [ ]:
#copy the entire above table in csv to datawrapper for the choropleth chart

In [ ]:
#Prices and Dynamics
comparison_states = [
    "New York",
    "United States",
    "California",
    "Connecticut",
    "Florida",
    "Massachusetts",
    "New Jersey",
    "Pennsylvania",
    "Texas",
    "Vermont"
]

date_cols = list(electricity.columns[3:-1])

latest_idx = len(date_cols) - 1

latest_month = date_cols[latest_idx]
previous_month = date_cols[latest_idx - 1]
year_ago_month = date_cols[latest_idx - 12]

month_name = latest_month[:3]
baseline_month = f"{month_name} 2019"

summary_rows = []

for state in comparison_states:

    row = electricity[
        electricity["State"] == state
    ].iloc[0]

    val_2019 = row[baseline_month]
    val_yoy = row[year_ago_month]
    val_mom = row[previous_month]
    val_curr = row[latest_month]

    summary_rows.append({
        "State": state,
        baseline_month: val_2019,
        year_ago_month: val_yoy,
        previous_month: val_mom,
        latest_month: val_curr,
        "MoM %": (val_curr / val_mom - 1) * 100,
        "YoY %": (val_curr / val_yoy - 1) * 100,
        "Since 2019 %": (val_curr / val_2019 - 1) * 100
    })

prices_dynamics = pd.DataFrame(summary_rows)

prices_dynamics[
    ["MoM %", "YoY %", "Since 2019 %"]
] = prices_dynamics[
    ["MoM %", "YoY %", "Since 2019 %"]
].round(2)

prices_dynamics = prices_dynamics.set_index("State").T

prices_dynamics

State,New York,United States,California,Connecticut,Florida,Massachusetts,New Jersey,Pennsylvania,Texas,Vermont
May 2019,17.35,13.31,18.82,23.34,11.50,22.28,16.78,14.25,12.01,17.41
May 2025,26.69,17.37,33.29,31.59,14.97,29.90,20.48,19.29,15.53,23.75
Apr 2026,29.45,18.83,35.25,32.24,15.38,29.45,23.53,21.47,16.99,24.56
May 2026,29.93,18.44,33.25,27.37,15.17,28.82,23.27,21.55,16.44,24.89
MoM %,1.63,-2.07,-5.67,-15.11,-1.37,-2.14,-1.10,0.37,-3.24,1.34
YoY %,12.14,6.16,-0.12,-13.36,1.34,-3.61,13.62,11.72,5.86,4.80
Since 2019 %,72.51,38.54,76.67,17.27,31.91,29.35,38.68,51.23,36.89,42.96


In [ ]:
#Long-Term Dynamics

long_term = (
    electricity[
        electricity["State"].isin([
            "United States",
            "New York"
        ])
    ]
    .set_index("State")[date_cols]
    .T
    .reset_index()
    .rename(
        columns={
            "index": "Date",
            "United States": "U.S."
        }
    )
)

long_term = long_term[
    ["Date", "U.S.", "New York"]
]

long_term["U.S."] = long_term["U.S."].round(2)
long_term["New York"] = long_term["New York"].round(2)

#display(long_term.style.hide(axis="index"))
long_term

State,Date,U.S.,New York
0,Jan 2001,7.73,13.89
1,Feb 2001,8.04,13.93
2,Mar 2001,8.32,13.58
3,Apr 2001,8.46,13.44
4,May 2001,8.83,14.01
...,...,...,...
300,Jan 2026,17.45,28.37
301,Feb 2026,17.65,29.99
302,Mar 2026,18.56,28.55
303,Apr 2026,18.83,29.45


In [ ]:
#converting datatype in columns
long_term["U.S."] = pd.to_numeric(long_term["U.S."], errors="coerce")
long_term["New York"] = pd.to_numeric(long_term["New York"], errors="coerce")


In [ ]:
# Difference Table

difference = long_term.copy()

difference["U.S."] = pd.to_numeric(
    difference["U.S."],
    errors="coerce"
)

difference["New York"] = pd.to_numeric(
    difference["New York"],
    errors="coerce"
)

difference["Difference"] = (
    difference["New York"]
    - difference["U.S."]
)

difference["Difference Percent"] = (
    difference["Difference"]
    / difference["U.S."]
    * 100
).round(1)


#display(difference.style.hide(axis="index"))
difference

State,Date,U.S.,New York,Difference,Difference Percent
0,Jan 2001,7.73,13.89,6.16,79.7
1,Feb 2001,8.04,13.93,5.89,73.3
2,Mar 2001,8.32,13.58,5.26,63.2
3,Apr 2001,8.46,13.44,4.98,58.9
4,May 2001,8.83,14.01,5.18,58.7
...,...,...,...,...,...
300,Jan 2026,17.45,28.37,10.92,62.6
301,Feb 2026,17.65,29.99,12.34,69.9
302,Mar 2026,18.56,28.55,9.99,53.8
303,Apr 2026,18.83,29.45,10.62,56.4


In [1]:
#Rolling 12M Difference in cents
#setup
rolling_avg = difference.copy()

rolling_avg["Rolling Average Difference"] = (
    rolling_avg["Difference"]
    .rolling(window=12)
    .mean()
)
rolling_avg = rolling_avg.dropna(
    subset=["Rolling Average Difference"]
)
rolling_table = pd.DataFrame(
    [
        rolling_avg["Rolling Average Difference"].round(2).tolist(),
        rolling_avg["Difference"].round(2).tolist()
    ],
    index=[
        "Rolling average difference in dollars",
        "Difference in dollars"
    ],
    columns=rolling_avg["Date"]
)

rolling_table

In [ ]:
#Rolling 12M Difference in percentage
rolling_avg_pct = difference.copy()

rolling_avg_pct["Rolling Average Percent"] = (
    difference["Difference Percent"]
    .rolling(window=12)
    .mean()
)

rolling_avg_pct = rolling_avg_pct.dropna(
    subset=["Rolling Average Percent"]
)
rolling_percent_table = pd.DataFrame(
    [
        rolling_avg_pct["Rolling Average Percent"].round(2).tolist(),
        rolling_avg_pct["Difference Percent"].round(2).tolist()
    ],
    index=[
        "Rolling average difference in percent",
        "Difference in percent"
    ],
    columns=rolling_avg_pct["Date"]
)

rolling_percent_table